<a href="https://colab.research.google.com/github/caiohenry-jpg/CalculoNumerico/blob/main/atividade_sistemas_lineares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
##Cálculo Numérico — Sistemas Lineares
##Caio Hênry - RA: 180374


import numpy as np

np.set_printoptions(precision=6, suppress=True)


def titulo(texto):
    print("\n" + "-" * 60)
    print(texto)
    print("-" * 60)


titulo("1. Forma matricial")

print("""
a) 2x + y = 7
   x - 3y = -5

Na forma AX = b:
""")

A = np.array([[2, 1], [1, -3]])
b = np.array([7, -5])

print("A =\n", A)
print("X = [x, y]ᵀ")
print("b =", b)

print("""
b) x + 2y - z = 2
   3x - y + 2z = 9
   2x + y + z = 7

Na forma AX = b:
""")

A = np.array([[1, 2, -1], [3, -1, 2], [2, 1, 1]])
b = np.array([2, 9, 7])

print("A =\n", A)
print("X = [x, y, z]ᵀ")
print("b =", b)

print("""
Em cada sistema, A contém os coeficientes das incógnitas,
X é o vetor das incógnitas e b contém os termos independentes.
""")


titulo("2. Eliminação de Gauss")

print("""
Matriz aumentada:

[  2   1  -1 |   8 ]
[ -3  -1   2 | -11 ]
[ -2   1   2 |  -3 ]

L2 <- L2 + (3/2)L1:

[  2    1   -1  |  8 ]
[  0   1/2  1/2 |  1 ]
[ -2    1    2  | -3 ]

L3 <- L3 + L1:

[ 2    1   -1  | 8 ]
[ 0   1/2  1/2 | 1 ]
[ 0    2    1  | 5 ]

L3 <- L3 - 4L2:

[ 2    1   -1  | 8 ]
[ 0   1/2  1/2 | 1 ]
[ 0    0   -1  | 1 ]

Substituição regressiva:

-z = 1
z = -1

y/2 + z/2 = 1
y/2 - 1/2 = 1
y = 3

2x + y - z = 8
2x + 3 + 1 = 8
x = 2

Solução: x = 2, y = 3 e z = -1.
""")


titulo("3. Número de soluções")

print("""
a) x + y = 3
   2x + 2y = 6

[ 1  1 | 3 ]
[ 2  2 | 6 ]

L2 <- L2 - 2L1:

[ 1  1 | 3 ]
[ 0  0 | 0 ]

A segunda linha resulta em 0 = 0. Resta apenas uma equação
para duas incógnitas, portanto há infinitas soluções.

Fazendo y = t:
x = 3 - t

Solução geral: (x, y) = (3 - t, t), com t real.
Sistema possível e indeterminado.
""")

print("""
b) x + y = 3
   2x + 2y = 7

[ 1  1 | 3 ]
[ 2  2 | 7 ]

L2 <- L2 - 2L1:

[ 1  1 | 3 ]
[ 0  0 | 1 ]

A segunda linha resulta em 0 = 1, que é uma contradição.
O sistema não possui solução.
Sistema impossível.
""")

print("""
c) x + y = 3
   2x - y = 0

[ 1   1 | 3 ]
[ 2  -1 | 0 ]

L2 <- L2 - 2L1:

[ 1   1 |  3 ]
[ 0  -3 | -6 ]

L2 <- (-1/3)L2:

[ 1  1 | 3 ]
[ 0  1 | 2 ]

L1 <- L1 - L2:

[ 1  0 | 1 ]
[ 0  1 | 2 ]

Solução única: x = 1 e y = 2.
Sistema possível e determinado.
""")


titulo("4. Implementação da eliminação de Gauss")


def gauss(A, b, mostrar=True, tol=1e-12):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)

    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A matriz deve ser quadrada e não vazia.")

    n = A.shape[0]

    if b.shape != (n,):
        raise ValueError("O tamanho de b deve ser igual ao número de linhas de A.")

    if not np.isfinite(A).all() or not np.isfinite(b).all():
        raise ValueError("A e b devem conter valores finitos.")

    if not np.isfinite(tol) or tol <= 0:
        raise ValueError("A tolerância deve ser positiva e finita.")

    # Montagem da matriz aumentada.
    M = np.column_stack((A, b))

    if mostrar:
        print("\nMatriz aumentada:")
        print(M)

    # Eliminação dos elementos abaixo da diagonal, sem trocar linhas.
    for k in range(n - 1):
        if abs(M[k, k]) < tol:
            raise ValueError(
                f"Pivô nulo ou próximo de zero na linha {k + 1}."
            )

        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            M[i, k:] -= m * M[k, k:]
            M[i, k] = 0.0

            if mostrar:
                print(f"\nL{i + 1} <- L{i + 1} - ({m:.6g})L{k + 1}")
                print(M)

    # Substituição regressiva.
    x = np.zeros(n)

    if mostrar:
        print("\nSubstituição regressiva:")

    for i in range(n - 1, -1, -1):
        if abs(M[i, i]) < tol:
            raise ValueError(
                f"Pivô nulo ou próximo de zero na linha {i + 1}."
            )

        soma = np.dot(M[i, i + 1:n], x[i + 1:])
        x[i] = (M[i, -1] - soma) / M[i, i]

        if mostrar:
            print(
                f"x{i + 1} = ({M[i, -1]:.6g} - ({soma:.6g}))"
                f" / ({M[i, i]:.6g}) = {x[i]:.6g}"
            )

    return x


A = np.array([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2]
], dtype=float)

b = np.array([8, -11, -3], dtype=float)

x_gauss = gauss(A, b)
x_numpy = np.linalg.solve(A, b)

print("\nSolução por Gauss:", x_gauss)
print("Solução por numpy.linalg.solve:", x_numpy)
print("Resultados iguais dentro da tolerância:", np.allclose(x_gauss, x_numpy))
print("Resíduo A @ x - b:", A @ x_gauss - b)

print("""
O resultado foi [2, 3, -1], igual ao obtido manualmente e pelo
NumPy. O resíduo é zero, confirmando que os valores satisfazem
as três equações.

Como não há pivoteamento, o algoritmo interrompe se encontrar
um pivô nulo ou próximo de zero. Isso não significa necessariamente
que o sistema não tenha solução; pode ser necessário trocar linhas.
""")


titulo("5. Balanço térmico — resolução manual")

print("""
4T1 - T2 = 15
-T1 + 4T2 - T3 = 10
-T2 + 3T3 = 10

Matriz aumentada:

[  4  -1   0 | 15 ]
[ -1   4  -1 | 10 ]
[  0  -1   3 | 10 ]

L2 <- L2 + (1/4)L1:

[ 4   -1    0 |  15  ]
[ 0   15/4 -1 | 55/4 ]
[ 0   -1    3 |  10  ]

L3 <- L3 + (4/15)L2:

[ 4   -1     0    |  15  ]
[ 0   15/4  -1    | 55/4 ]
[ 0    0    41/15 | 41/3 ]

Substituição regressiva:

(41/15)T3 = 41/3
T3 = (41/3) * (15/41) = 5

(15/4)T2 - T3 = 55/4
(15/4)T2 = 55/4 + 5 = 75/4
T2 = 5

4T1 - T2 = 15
4T1 = 20
T1 = 5

Solução: T1 = T2 = T3 = 5.
""")


titulo("5. Balanço térmico — solução em Python")

A = np.array([
    [4, -1, 0],
    [-1, 4, -1],
    [0, -1, 3]
], dtype=float)

b = np.array([15, 10, 10], dtype=float)

T = gauss(A, b)
T_numpy = np.linalg.solve(A, b)

print("\nTemperaturas:")
for i, temperatura in enumerate(T, start=1):
    print(f"T{i} = {temperatura:.6f}")

print("\nResultado do NumPy:", T_numpy)
print("Resultados iguais dentro da tolerância:", np.allclose(T, T_numpy))
print("Resíduo A @ T - b:", A @ T - b)

print("""
As três temperaturas são iguais a 5, na unidade adotada pelo
modelo. O enunciado não informa a unidade.

Substituindo nas equações:

4(5) - 5 = 15
-5 + 4(5) - 5 = 10
-5 + 3(5) = 10

Os valores satisfazem os três balanços. A solução manual
e os resultados calculados em Python coincidem.
""")


------------------------------------------------------------
1. Forma matricial
------------------------------------------------------------

a) 2x + y = 7
   x - 3y = -5

Na forma AX = b:

A =
 [[ 2  1]
 [ 1 -3]]
X = [x, y]ᵀ
b = [ 7 -5]

b) x + 2y - z = 2
   3x - y + 2z = 9
   2x + y + z = 7

Na forma AX = b:

A =
 [[ 1  2 -1]
 [ 3 -1  2]
 [ 2  1  1]]
X = [x, y, z]ᵀ
b = [2 9 7]

Em cada sistema, A contém os coeficientes das incógnitas,
X é o vetor das incógnitas e b contém os termos independentes.


------------------------------------------------------------
2. Eliminação de Gauss
------------------------------------------------------------

Matriz aumentada:

[  2   1  -1 |   8 ]
[ -3  -1   2 | -11 ]
[ -2   1   2 |  -3 ]

L2 <- L2 + (3/2)L1:

[  2    1   -1  |  8 ]
[  0   1/2  1/2 |  1 ]
[ -2    1    2  | -3 ]

L3 <- L3 + L1:

[ 2    1   -1  | 8 ]
[ 0   1/2  1/2 | 1 ]
[ 0    2    1  | 5 ]

L3 <- L3 - 4L2:

[ 2    1   -1  | 8 ]
[ 0   1/2  1/2 | 1 ]
[ 0    0   -1  | 1 ]

Substituição